In [1]:
# === Setup en verbinding ===
import ConnectionConfig as cc
from pyspark.sql.functions import (
    col, year, month, dayofmonth, weekofyear, date_format,
    weekday, when, expr, to_date, row_number
)
from pyspark.sql.window import Window

debugging_mode = True


In [2]:
#  Extract
cc.setupEnvironment()
cc.setupEnvironment()
print(cc.config.sections())

spark = cc.startLocalCluster("DIM_DATE", 4)
spark.getActiveSession()
cc.config.read('config.ini')
cc.set_connectionProfile("catchem")


Environment variables are set...
Environment variables are set...
['default', 'tutorial_op', 'catchem', 'kafka']


In [3]:
#EXTRACT
date_src = (
    spark.read
        .format("jdbc")
        .option("url", cc.create_jdbc())
        .option("driver", cc.get_Property("driver"))
        .option(
            "dbtable",
            "(select id, log_type, log_time from treasure_log) as subq"
        )
        .option("user", cc.get_Property("username"))
        .option("password", cc.get_Property("password"))
        .load()
        .filter(col("log_type") == 2)
)

if debugging_mode:
    print("Preview van date_src (extract):")
    date_src.show(5)

Preview van date_src (extract):
+--------------------+--------+--------------------+
|                  id|log_type|            log_time|
+--------------------+--------+--------------------+
|[FF FE 0C 6B 6C D...|       2|2022-03-25 20:33:...|
|[FF FE 1C B8 23 7...|       2|2021-11-17 17:57:...|
|[FF FE 1E FE CD 2...|       2|2020-09-20 21:37:...|
|[FF FE 24 D1 EF C...|       2|2023-07-17 08:11:...|
|[FF FE 27 18 5B 5...|       2|2023-02-21 11:29:...|
+--------------------+--------+--------------------+
only showing top 5 rows


In [4]:
# TRANSFORM
neededDates = (
    date_src
        .withColumn("calendarDate", to_date(col("log_time")))
        .select("calendarDate")
        .distinct()
        .orderBy("calendarDate")
)
if debugging_mode:
    print("Unieke datums uit log_time (transform):")
    neededDates.show(10)

Unieke datums uit log_time (transform):
+------------+
|calendarDate|
+------------+
|  2020-09-11|
|  2020-09-12|
|  2020-09-13|
|  2020-09-14|
|  2020-09-15|
|  2020-09-16|
|  2020-09-17|
|  2020-09-18|
|  2020-09-19|
|  2020-09-20|
+------------+
only showing top 10 rows


In [5]:
# TRANSFORM
windowSpec = Window.orderBy("calendarDate")

dimDate = (
    neededDates
        .withColumn("DateSurKey", expr("uuid()"))
        .withColumn("DateId", row_number().over(windowSpec))
        .withColumn("Day", dayofmonth(col("calendarDate")))
        .withColumn("Week", weekofyear(col("calendarDate")))
        .withColumn("Month", date_format(col("calendarDate"), "MMMM"))
        .withColumn("Year", year(col("calendarDate")))
        .withColumn("MonthOfTheYear", month(col("calendarDate")))
        .withColumn("DayOfTheWeek", weekday(col("calendarDate")) + 1)
        .withColumn("IsWeekDay", when(weekday(col("calendarDate")) < 5, True).otherwise(False))
        .withColumn("calendarDate", col("calendarDate"))
        .select(
            "DateSurKey",
            "DateId",
            "Day",
            "Week",
            "Month",
            "Year",
            "MonthOfTheYear",
            "DayOfTheWeek",
            "IsWeekDay",
            "calendarDate"
        )
)

if debugging_mode:
    print("DimDate (transform) preview:")
    dimDate.show(10)


DimDate (transform) preview:
+--------------------+------+---+----+---------+----+--------------+------------+---------+------------+
|          DateSurKey|DateId|Day|Week|    Month|Year|MonthOfTheYear|DayOfTheWeek|IsWeekDay|calendarDate|
+--------------------+------+---+----+---------+----+--------------+------------+---------+------------+
|6b5c7721-85bb-44e...|     1| 11|  37|September|2020|             9|           5|     true|  2020-09-11|
|8d0113ed-8935-451...|     2| 12|  37|September|2020|             9|           6|    false|  2020-09-12|
|30829b78-1aca-400...|     3| 13|  37|September|2020|             9|           7|    false|  2020-09-13|
|6abf55a0-442b-4af...|     4| 14|  38|September|2020|             9|           1|     true|  2020-09-14|
|7d05d407-3d0e-414...|     5| 15|  38|September|2020|             9|           2|     true|  2020-09-15|
|d75e0bee-f793-478...|     6| 16|  38|September|2020|             9|           3|     true|  2020-09-16|
|0707b2de-3131-43a...|    

In [6]:
#  LOAD
dimDate.write.format("delta").mode("overwrite").save("delta/DATE_DIM")

if debugging_mode:
    print("DimDate succesvol opgeslagen naar delta/DATE_DIM")


DimDate succesvol opgeslagen naar delta/DATE_DIM


In [7]:
spark.stop()